# Classification de textes – détecteur de spams

Nous commençons par charger toutes les paquets Python et fonctions auxiliaires nécessaires.

In [1]:
# Load packages and helper functions
%run data_prep_tools.py
%run EDA_tools.py

# Données

Nous utiliserons le corpus d'e-mails public de [SpamAssassin](https://spamassassin.apache.org/). Ce jeu de données contient environ 6 000 e-mails étiquetés. Pour en savoir plus, consultez [cette page](https://spamassassin.apache.org/old/publiccorpus/). (*Note : en traitement de données textuelles, on parle de « corpus » pour désigner l'ensemble des données, et de « documents » pour désigner les éléments individuels.*)

Le jeu de données a été téléchargé pour vous et est disponible dans le dossier *data*.

Les données sont étiquetées : chaque e-mail est identifié soit comme un « spam » (par exemple, s'il a été signalé comme tel par un utilisateur), soit comme un e-mail légitime (non-spam, aussi appelé « ham »).

Notre objectif est d'explorer et de comparer différents espaces de caractéristiques ainsi que diverses approches d'apprentissage automatique (*machine learning*). L'utilisation d'e-mails (spams et e-mails légitimes) sert ici uniquement à des fins de démonstration et d'apprentissage ; il s'agit d'un exemple textuel familier à tous, qui permet d'illustrer les différentes étapes du développement d'une application d'apprentissage automatique, ainsi que les processus décisionnels qui jalonnent ce parcours.

## Préparation des données – Aperçu

Dans ce notebook, nous explorerons le jeu de données, effectuerons une première analyse et le préparerons en vue de diverses tâches d'apprentissage automatique (machine learning).

### Tâche

Nous traiterons les données brutes, nettoierons le texte et extrairons des caractéristiques (features) supplémentaires afin de préparer les données pour une analyse approfondie et pour la construction de nos modèles d'apprentissage automatique.

### Aperçu du notebook

* Chargement des données
* Prétraitement du texte
* Création de caractéristiques
* Enregistrement des données nettoyées

## Chargement de nos données

In [2]:
# Load the data
df_source = load_source_data()

8546 emails loaded
Cleaning data set:
2710 duplicate emails found and removed
4 empty emails found and removed

5832 emails remaining

Number of columns: 2
Columns names:
spam_label, text


<div class="alert alert-success">
<h3>Questions</h3>

Observez l'exemple dans la cellule suivante et essayez de répondre aux questions ci-dessous :

__Q1.__ Quelles parties du texte considérez-vous comme du « bruit » ?

__Q2.__ Que devrions-nous faire de ces parties du texte ?
</div>

In [3]:
# If you rerun this cell multiple times you get different samples displayed each time
# OR you can replace the number 3 with a number of your choice
display(df_source.sample(3))

,spam_label,text
4969,0,">I currently have spam being filtered for about 50 users via spamd. >Works perfectly! However, is there anyone out there providing reports >to their users about what was filetered? Maybe just something like >date/time, From, and subject? Yup! You can check: http://www.cs.ait.ac.th/laboratory/email/quarantine.shtml Though it uses procmail! Olivier ------------------------------------------------------- This sf.net email is sponsored by:ThinkGeek Welcome to geek heaven. http://thinkgeek.com/sf _______________________________________________ Spamassassin-talk mailing list Spamassassin-talk@lists.sourceforge.net https://lists.sourceforge.net/lists/listinfo/spamassassin-talk"
5101,0,"use Perl Daily Headline Mailer Damian Conway to speak in London posted by ziggy on Thursday August 08, @15:21 (damian) http://use.perl.org/article.pl?sid=02/08/08/1923211 Meeting in Budapest posted by ziggy on Thursday August 08, @17:24 (groups) http://use.perl.org/article.pl?sid=02/08/08/1927253 Copyright 1997-2002 pudge. All rights reserved. ====================================================================== You have received this message because you subscribed to it on use Perl. To stop receiving this and other messages from use Perl, or to add more messages or change your preferences, please go to your user page. http://use.perl.org/my/messages/ You can log in and change your preferences from there."
539,0,"<HTML><HEAD><TITLE>Buyer's Alert | July 18, 2002</TITLE> </HEAD> <body style=""margin:8px 9px 9px 12px"" bgcolor=""#ffffff"" background=""http://techupdate.zdnet.com/techupdate/i/bg_232850.gif"" link=""#003399"" alink=""#cc0000"" vlink=""#666699""> <CENTER><A name=top></A> <TABLE cellSpacing=0 cellPadding=0 width=612 bgColor=#232850 border=0> <TBODY> <TR vAlign=bottom> <TD width=440 colSpan=4><IMG height=60 alt=""ZDNet Buyers Alert Newsletter"" src=""http://www.cnet.com/i/sh/nl/hd_buyers_alert.gif"" width=440></TD> <TD vAlign=top align=middle width=160 bgColor=#ffffff rowSpan=2><!-- ad --> <div align=""center""><img src=""http://i.i.com.com/cnwk.1d/Ads/common/advertisement.gif"" width=""120"" height=""10"" border=""0""></br> <a href=""http://clickthru.online.com/Click?q=fc-WpHmQVwTauRU4pZa9-0t3VVat9yR"" target=""new""> <img src=""http://www.techrepublic.com/images/ads/Toshiba.sky.2400freebag_160x600.gif"" width=""160"" height=""600"" border=""0""> </a></div> <!-- /ad --><!-- rebates --> <br><img src=""http://www.cnet.com/i/nl/zd-ba/rebates.gif"" width=160 height=18 alt=""Rebates""><br> <table width=160 bgcolor=""#eeeeee"" cellpadding=5 cellspacing=0 border=0> <tr><td><font face=""arial, helvetica"" size=""-2"">When ordering, make sure the reseller/manufacturer provides the rebate coupon you need. <br><br> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires July 31, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=11-l8riI2-aPLZhppqX3VynhNBuCqlR"" >Get a car cassette adapter and power adapter FREE with Rio Volt SP250</a> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires August 31, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=27-z9ZmIqI14RundZFSkMRUVolYrBeR"" >$20 off SideWinder Strategic Commander</a> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires September 30, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=3c-tK4RIcIw_GwZ-o1Y0VXXjhrHNHyR"" >$20 off Rio 600 64MB</a> <li><a href=""http://clickthru.online.com/Click?q=51-QgUWIjJWhgT2xbBJlh9GpMUHM3FR"" >$20 off Nik..."


## Espace pour vos observations :

1. Certains éléments présents dans le texte devraient être supprimés ou transformés afin de le rendre lisible et exploitable. 
Voici quelques suggestions :
* Balises HTML
* URL
* Adresses e-mail
* Signes de ponctuation, chiffres (par ex. 2002, 1,1, ...)
* Espaces multiples
* Normalisation de la casse (par ex. « Dog » vs « dog », ...)
* Mots vides (« stopwords ») en anglais (par ex. *a, is, my, i, all, and, by*...)
* ...

2. L'expérience montre que le nombre d'occurrences des éléments susmentionnés (balises HTML, URL, etc.) peut s'avérer utile pour distinguer les spams des messages légitimes. De même, la longueur des e-mails ainsi que la fréquence des signes de ponctuation ou des lettres majuscules peuvent fournir des indices permettant de déterminer s'il s'agit ou non d'un spam.

## Nettoyage et prétraitement du texte

Un prétraitement efficace du texte constitue une étape essentielle de tout projet de traitement du langage naturel (NLP). Il s'agit de la première phase du pipeline d'apprentissage automatique ; il est donc crucial de la réaliser correctement. L'objectif du prétraitement est de transformer le texte brut en un format exploitable par les algorithmes d'apprentissage automatique.

Notre objectif global est de concevoir des modèles capables de distinguer les messages légitimes des courriers indésirables (spams).

Les exemples précédents montrent que certaines données sont particulièrement désordonnées et contiennent une grande quantité d'éléments inutiles à la compréhension humaine — ce que l'on appelle du « bruit ». Dans un premier temps, nous allons **nettoyer** et **normaliser** le texte brut. L'idée est de conserver un maximum de mots **informatifs** tout en éliminant ceux qui ne le sont pas. Supprimer ce « bruit » permettra d'améliorer la précision de nos modèles.

Il nous faut donc identifier les éléments qui constituent ce « bruit » au sein du texte, afin de les retirer.

La fonction *clean_corpus* ci-dessous prendra en charge les points soulevés dans la première série d’observations.

In [4]:
df_cleaned = clean_corpus(df_source)

Number of samples: 5832
Number of columns: 3
Columns names:
spam_label, text, text_cleaned

Number of duplicate cleaned texts found: 280
Number of empty texts found: 28

Email texts cleaned
Number of samples: 5832


In [5]:
# Let's look at some examples.
# You can rerun this cell to get a different sample
show_clean_text(df_cleaned)


Original document:

> Date: Mon, 30 Sep 2002 14:00:12 -0400 (EDT) > From: Dayv Gastonguay <noghri@nauticom.net> > > I
just installed razor 2.152 on a FreeBSD 4.4-RELEASE box and having > problems with razor-check. Any
time razor-check is run, (with or without > arguments), i get this error: > > Can't use an undefined
value as a symbol reference at >
/usr/local/lib/perl5/site_perl/5.005/i386-freebsd/Razor2/Client/Agent.pm > line 756. > Try
installing the latest Perl (at least 5.6.1) port on Freebsd and make sure you set the system to use
perl from ports (i.e. in the ports/lang/perl5/files directory run ./use.perl port. Reinstall the
relevant perl modules needed by razor and try again. Sven
------------------------------------------------------- This sf.net email is sponsored by:ThinkGeek
Welcome to geek heaven. http://thinkgeek.com/sf _______________________________________________
Razor-users mailing list Razor-users@lists.sourceforge.net
https://lists.sourceforge.net/lists/listinfo/r

<div class="alert alert-info">
<h2>Ingénierie des caractéristiques (Feature engineering)</h2>

Nous présentons ci-dessous trois méthodes différentes pour extraire des informations à partir des données. Il s'agit de trois représentations distinctes des données, à partir desquelles les modèles peuvent potentiellement apprendre :
1.  **Caractéristiques numériques** : nous nous intéressons uniquement à la fréquence des « signatures caractéristiques du spam ». Aucune information relative aux mots contenus dans les e-mails n'est utilisée.
2.  **Comptage de mots** : nous examinons la fréquence d'apparition des différents mots dans chaque texte. Nous procédons de deux manières : d'abord en notant simplement si un mot « apparaît dans l'e-mail », puis en calculant une fréquence relative (en comparant la fréquence d'un mot dans un e-mail donné par rapport à l'ensemble des autres e-mails).
3.  Nous utilisons un modèle de langage étendu (LLM) existant pour transformer le texte de l'e-mail en une nouvelle représentation, plus efficace et « enrichie en informations », appelée *embedding* (plongement vectoriel).

</div>


## Partie 1 : Extraction de caractéristiques numériques

Nous partons des idées issues de la deuxième observation et créons de nouvelles caractéristiques qui quantifient la présence de différentes composantes de bruit dans le texte.

In [6]:
num_features_df = extract_numeric_features(df=df_source, with_labels=True)

Number of samples and columns of input: (5832, 2)
Number of columns: 2
Columns names:
spam_label, text

Numeric features extracted
Data size: (5832, 14)
Number of columns: 14
Columns names:
email_counts, html tag_counts, url_counts, Twitter username_counts, hashtag_counts
character_counts, word_counts, unique word_counts, punctuation mark_counts, uppercase word_counts
lowercase word_counts, digit_counts, alphabetic char_counts, spam_label
Numeric features saved to data/num_features.csv


## Part 2: Extracting features from text

## Partie 2 : Extraction de caractéristiques à partir de texte

Les ordinateurs ne comprennent pas le langage naturel, qui se présente sous une forme non structurée. Alors, comment représenter le texte ?

### Sac de mots (Bag of Words)

L'un des modèles les plus simples — mais qui, aux débuts du traitement du langage naturel (NLP), s'est avéré efficace et couramment utilisé — pour représenter du texte en vue d'un apprentissage automatique est le modèle du « sac de mots » ([*Bag of Words*](https://en.wikipedia.org/wiki/Bag-of-words_model)). Avec ce modèle, on écarte la majeure partie de la structure du texte d'origine (ordre des mots, chapitres, paragraphes, phrases ou mise en forme) pour ne conserver que la fréquence d'apparition de chaque mot dans chaque texte. Le fait d'ignorer la structure et de ne compter que les occurrences des mots conduit à l'image mentale d'un texte représenté comme un « sac ».

**Exemple :** Imaginons un corpus simplifié composé de quatre documents.

```python
corpus = [
    'I enjoy paragliding.',
    'I do like NLP.',
    'I like deep learning.',
    'O Captain! my Captain!'
]
```

In [7]:
show_bag_of_words_vector()

,captain,deep,do,enjoy,i,learning,like,my,nlp,o,paragliding
Text,,,,,,,,,,,
I enjoy paragliding.,0,0,0,1,1,0,0,0,0,0,1
I do like NLP.,0,0,1,0,1,0,1,0,1,0,0
I like deep learning.,0,1,0,0,1,1,1,0,0,0,0
O Captain! my Captain!,2,0,0,0,0,0,0,1,0,1,0


Dans le tableau ci-dessus, chaque colonne représente un mot du corpus et chaque ligne, l'un des quatre documents. La valeur contenue dans chaque cellule indique le nombre d'occurrences d'un mot donné dans un document spécifique. Par exemple, dans le quatrième document, le mot « captain » apparaît deux fois, tandis que les mots « my » et « O » y figurent chacun une fois.

L'implémentation technique de l'approche « sac de mots » (Bag of Words) est appelée `CountVectorizer`. Elle convertit chaque document en une suite de nombres, c'est-à-dire un vecteur numérique — d'où le terme « vectorizer » (vectoriseur).

Bien que ce type de transformation permette aux algorithmes d'apprentissage automatique de traiter efficacement des données textuelles, elle présente un inconvénient : elle considère tous les mots comme indépendants, faisant abstraction du contexte dans lequel ils apparaissent. Par exemple, la perte d'informations sur l'ordre des mots peut altérer le sens d'une phrase. Les phrases « I do like NLP », « Do I like NLP » ou encore « NLP like I do » contiennent les mêmes mots, mais possèdent des significations différentes.

### TF-IDF

L'approche **TF-IDF** (pour *Term Frequency–Inverse Document Frequency*, ou « fréquence du terme vs fréquence inverse dans les documents ») vise à pallier cette limite en mesurant l'importance d'un mot au sein d'un document, relativement à l'ensemble des documents (le corpus).

Nous utilisons ici l'implémentation proposée par `scikit-learn`. Le score TF-IDF est calculé comme le produit des éléments suivants :
- La **fréquence du terme (TF)**, qui correspond au rapport entre la fréquence du mot $w$ dans un document donné $d$ et le nombre total de mots dans ce même document :
$TF(w, d) = \frac{f(w, d)}{N(d)}$
- La **fréquence inverse dans les documents (IDF)** — ici sous une forme « lissée » —, définie par la formule :
$$IDF(w, D) = \log\left(\frac{size(D)+1}{df(w, D)+1}\right)+1$$
où $df(w, D)$ représente le nombre de documents, au sein du corpus $D$, qui contiennent le mot $w$. L'ajout de « 1 » au numérateur et au dénominateur permet de garantir que la valeur IDF reste finie et stable. Ainsi, les mots courants qui apparaissent dans de nombreux documents (faible IDF) se voient accorder un poids moindre, tandis que les mots rares qui n'apparaissent que dans un petit nombre de documents se voient attribuer un poids plus élevé (IDF élevé).

In [8]:
show_tfidf_vector()

,captain,deep,do,enjoy,i,learning,like,my,nlp,o,paragliding
Text,,,,,,,,,,,
I enjoy paragliding.,0.000000,0.00000,0.00000,0.644503,0.411378,0.00000,0.000000,0.000000,0.00000,0.000000,0.644503
I do like NLP.,0.000000,0.00000,0.57458,0.000000,0.366747,0.00000,0.453005,0.000000,0.57458,0.000000,0.000000
I like deep learning.,0.000000,0.57458,0.00000,0.000000,0.366747,0.57458,0.453005,0.000000,0.00000,0.000000,0.000000
O Captain! my Captain!,0.816497,0.00000,0.00000,0.000000,0.000000,0.00000,0.000000,0.408248,0.00000,0.408248,0.000000


Vous pouvez extraire les caractéristiques textuelles ci-dessous en utilisant soit `CountVectorizer` (`vectorizer="count"`), soit `TfidfVectorizer` (`vectorizer="tfidf"`). Veuillez noter que cette opération peut prendre un certain temps en fonction de la mémoire disponible sur votre machine ; faites donc preuve de patience.

C'est pourquoi nous avons déjà précalculé ces caractéristiques à l'aide de la méthode `"tfidf"` et les avons stockées dans le dossier `features`. Vous pouvez les charger en utilisant la commande `load_feature_space(features="text")`.

In [9]:
%run data_prep_tools.py

text_features_df = extract_text_features(
    df_cleaned, vectorizer="tfidf", with_labels=True, store=True
)
text_features_df.shape

TF-IDF Vectorizer selected
Corpus data type:  <class 'scipy.sparse._csr.csr_matrix'>
0.8745627968449597 % are null values
Number of columns: 8219
First 5 names:
aall, aaron, abacha, abandon, abandoned
Last 5 columns:
zoom, zope, zurich, zyban, spam_label
Text features saved to  data/sparse_text_features_tfidf.npz


(5832, 8219)

## Partie 3 : Utiliser des LLM pour représenter le texte sous forme de nombres

### Plongements lexicaux (Embeddings)

Les approches de type « sac de mots » (Bag of Words) et TF-IDF ne permettent pas de saisir le sens des mots ni les relations qui les unissent. De plus, elles aboutissent à des représentations textuelles de très grande dimension et très creuses (parsemées de zéros), ce qui manque d'efficacité et peut entraîner un surapprentissage (overfitting).
Pour pallier ces limites, on peut recourir à des **plongements lexicaux** (ou *embeddings*) ou à des modèles basés sur l'architecture « Transformer ». Les *embeddings* sont des représentations vectorielles plus denses des mots, apprises à partir de vastes corpus textuels. En associant des vecteurs similaires à des mots ayant un sens proche, ils permettent de capturer le sens et les relations au sein d'un espace vectoriel continu de dimension plus réduite.

Nous avons traité les textes des e-mails via un modèle de langage afin de générer les *embeddings* correspondants. Comme l'extraction de ces caractéristiques est une opération chronophage, nous avons sauvegardé ces *embeddings* et les avons mis à votre disposition dans le fichier nommé `email_embeddings.csv`.

Vous pouvez les charger en utilisant la commande `load_feature_space(features="embeddings")`.

In [10]:
embeddings_df = load_feature_space(features="embedding")

Email embeddings loaded
Data includes labels in the column 'spam_label'
The data set has 5832 rows, 769 columns
